visualization script with real figures sourced
directly from verification_results.csv, semantic_rag_report.json, and
threshold_sweep_results.csv — every number here is traceable to an actual run.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import json
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')

INTERMEDIATE_DIR = Path("../Datasets/intermediate")
OUTPUT_DIR = Path("../figures")
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
# ============================================================================
# Load real data
# ============================================================================
results_df = pd.read_csv(INTERMEDIATE_DIR / "verification_results.csv")

with open(INTERMEDIATE_DIR / "semantic_rag_report.json") as f:
    semantic_report = json.load(f)

sweep_df = pd.read_csv(INTERMEDIATE_DIR / "threshold_sweep_results.csv")

BEST_TC, BEST_TF = 0.50, 0.55

def recompute_decision(row, tc, tf):
    if row['C_f'] < tc:
        return 'HALLUCINATION'
    if row['patterns_detected'] != 'None' and pd.notna(row['patterns_detected']):
        return 'HALLUCINATION'
    if row['Conf_f'] < tf:
        return 'HALLUCINATION'
    return 'VERIFIED'

results_df['decision_best'] = results_df.apply(
    lambda r: recompute_decision(r, BEST_TC, BEST_TF), axis=1
)

In [3]:
def compute_metrics(df, decision_col):
    TP = len(df[(df.ground_truth == 'HALLUCINATION') & (df[decision_col] == 'HALLUCINATION')])
    TN = len(df[(df.ground_truth == 'VERIFIED') & (df[decision_col] == 'VERIFIED')])
    FP = len(df[(df.ground_truth == 'VERIFIED') & (df[decision_col] == 'HALLUCINATION')])
    FN = len(df[(df.ground_truth == 'HALLUCINATION') & (df[decision_col] == 'VERIFIED')])
    total = len(df)
    acc = (TP + TN) / total if total else 0
    prec = TP / (TP + FP) if (TP + FP) else 0
    rec = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

In [4]:
msgr_metrics = compute_metrics(results_df, 'decision_best')
semantic_metrics = semantic_report['overall_performance']

In [5]:
# ============================================================================
# Figure 1: MSGR-RAG vs Semantic RAG — overall performance
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 6))
metrics_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
msgr_vals = [msgr_metrics['accuracy']*100, msgr_metrics['precision']*100,
             msgr_metrics['recall']*100, msgr_metrics['f1']*100]
semantic_vals = [semantic_metrics['accuracy']*100, semantic_metrics['precision']*100,
                 semantic_metrics['recall']*100, semantic_metrics['f1']*100]

x = np.arange(len(metrics_labels))
width = 0.35
bars1 = ax.bar(x - width/2, msgr_vals, width, label='MSGR-RAG', color='#2E86AB', alpha=0.85, edgecolor='black')
bars2 = ax.bar(x + width/2, semantic_vals, width, label='Semantic RAG (baseline)', color='#A23B72', alpha=0.85, edgecolor='black')

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%', ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Score (%)', fontsize=13, fontweight='bold')
ax.set_title(f'MSGR-RAG vs. Semantic RAG Baseline\n(n={len(results_df)} facts, θ_consistency={BEST_TC}, θ_confidence={BEST_TF})',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig1_msgr_vs_semantic.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig1_msgr_vs_semantic.png")

Saved fig1_msgr_vs_semantic.png


In [6]:
# ============================================================================
# Figure 2: Per-category accuracy — MSGR-RAG vs Semantic RAG
# ============================================================================
fig, ax = plt.subplots(figsize=(11, 6))

msgr_cat = {}
for cat in results_df['category'].unique():
    cat_df = results_df[results_df.category == cat]
    msgr_cat[cat] = (cat_df.ground_truth == cat_df.decision_best).mean() * 100

semantic_cat = {k: v['accuracy']*100 for k, v in semantic_report['category_breakdown'].items()}

categories = ['nonexistent_relationship', 'wrong_predicate', 'domain_violation', 'real_edge']
cat_display = ['Nonexistent\nRelationship', 'Wrong\nPredicate', 'Domain\nViolation\n(central thesis)', 'Real\nEdge']

msgr_c_vals = [msgr_cat[c] for c in categories]
semantic_c_vals = [semantic_cat[c] for c in categories]

x = np.arange(len(categories))
bars1 = ax.bar(x - width/2, msgr_c_vals, width, label='MSGR-RAG', color='#2E86AB', alpha=0.85, edgecolor='black')
bars2 = ax.bar(x + width/2, semantic_c_vals, width, label='Semantic RAG', color='#A23B72', alpha=0.85, edgecolor='black')

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('Per-Category Detection Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cat_display, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_per_category_accuracy.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig2_per_category_accuracy.png")

Saved fig2_per_category_accuracy.png


In [7]:
# ============================================================================
# Figure 3: Threshold sensitivity heatmap (F1 across theta_consistency x theta_confidence)
# ============================================================================
fig, ax = plt.subplots(figsize=(9, 7))
pivot = sweep_df.pivot(index='theta_confidence', columns='theta_consistency', values='f1')
im = ax.imshow(pivot.values, cmap='viridis', aspect='auto', origin='lower')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{v:.2f}' for v in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{v:.2f}' for v in pivot.index])
ax.set_xlabel('θ_consistency', fontsize=12, fontweight='bold')
ax.set_ylabel('θ_confidence', fontsize=12, fontweight='bold')
ax.set_title('F1-Score Sensitivity to Decision Thresholds', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('F1-Score', fontsize=11)

# mark the chosen operating point
tc_idx = list(pivot.columns).index(BEST_TC)
tf_idx = list(pivot.index).index(BEST_TF)
ax.plot(tc_idx, tf_idx, 'r*', markersize=20, markeredgecolor='white', markeredgewidth=1.5)
ax.annotate('Selected\noperating point', xy=(tc_idx, tf_idx), xytext=(tc_idx+1, tf_idx+1),
            fontsize=10, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig3_threshold_sensitivity.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig3_threshold_sensitivity.png")

Saved fig3_threshold_sensitivity.png


In [8]:
# ============================================================================
# Figure 4: Confusion matrix at chosen threshold
# ============================================================================
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 6))
TP = len(results_df[(results_df.ground_truth == 'HALLUCINATION') & (results_df.decision_best == 'HALLUCINATION')])
TN = len(results_df[(results_df.ground_truth == 'VERIFIED') & (results_df.decision_best == 'VERIFIED')])
FP = len(results_df[(results_df.ground_truth == 'VERIFIED') & (results_df.decision_best == 'HALLUCINATION')])
FN = len(results_df[(results_df.ground_truth == 'HALLUCINATION') & (results_df.decision_best == 'VERIFIED')])

cm = np.array([[TP, FN], [FP, TN]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['HALL', 'VERIF'],
            yticklabels=['HALL', 'VERIF'], cbar_kws={'label': 'Count'}, ax=ax, annot_kws={'fontsize': 14})
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title(f'Confusion Matrix (n={len(results_df)}, θ_c={BEST_TC}, θ_conf={BEST_TF})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig4_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig4_confusion_matrix.png")

print("\nAll figures generated from real data.")

Saved fig4_confusion_matrix.png

All figures generated from real data.
